In [1]:
import numpy as np
import pandas as pd
import random
import re
from sklearn.model_selection import train_test_split
import json
import ast
import re

In [ ]:
#Helper functions to sanitize the code snippets

def sanitize_code(code):

    code = code[len("```python")+1:-len("```")].strip()
    
    # Remove single-line comments
    code = re.sub(r'# .*', '', code)
    
    return code

#Extract the portions for each datapoint
def extract_gpt_output(string):
    terms = ["VUL:","REASON:","SEC:","INST:"]

    for item in terms:
        if item not in string:
            return -1
        
    vulnerable_code_pos = string.find(terms[0])
    secure_code_pos = string.find(terms[2])
    instruction_pos = string.find(terms[3])

    reasoning_pos = [m.start() for m in re.finditer(terms[1], string)]

    vulnerable_code_reasoning_pos = reasoning_pos[0]
    secure_code_reasoning_pos = reasoning_pos[1]

    vulnerable_code = string[vulnerable_code_pos:vulnerable_code_reasoning_pos]
    vulnerable_code = vulnerable_code[len(terms[0])+1:].strip()

    vulnerable_code_reasoning = string[vulnerable_code_reasoning_pos:secure_code_pos]
    vulnerable_code_reasoning = vulnerable_code_reasoning[len(terms[1])+1:].strip()

    secure_code = string[secure_code_pos:secure_code_reasoning_pos]
    secure_code = secure_code[len(terms[2])+1:].strip()

    secure_code_reasoning = string[secure_code_reasoning_pos:instruction_pos]
    secure_code_reasoning = secure_code_reasoning[len(terms[1])+1:].strip()

    instruction = string[instruction_pos:]
    instruction = instruction[len(terms[3])+1:].strip()

    return{
        "Vulnerable Code":sanitize_code(vulnerable_code),
        "Secure Code":sanitize_code(secure_code),
        "Vulnerable Code Reasoning":vulnerable_code_reasoning,
        "Secure Code Reasoning":secure_code_reasoning,
        "Instruction":instruction
    }

def parsed_dataframe(array):
    vulnerable_code_list = []
    secure_code_list = []
    vulnerable_code_reasoning_list = []
    secure_code_reasoning_list = []
    instruction_list = []
    for idx,item in enumerate(array):
        extracted_point = extract_gpt_output(item)
        if(extracted_point==-1):
            print(idx,item)
            continue
        else:
            vulnerable_code_list.append(extracted_point["Vulnerable Code"])
            secure_code_list.append(extracted_point["Secure Code"])
            vulnerable_code_reasoning_list.append(extracted_point["Vulnerable Code Reasoning"])
            secure_code_reasoning_list.append(extracted_point["Secure Code Reasoning"])
            instruction_list.append(extracted_point["Instruction"])
    
    new_df = pd.DataFrame({
        "Vulnerable Code":vulnerable_code_list,
        "Secure Code":secure_code_list,
        "Vulnerable Code Reasoning":vulnerable_code_reasoning_list,
        "Secure Code Reasoning":secure_code_reasoning_list,
        "Instruction":instruction_list})
    
    return new_df


def train_val_test_split(df, train_frac=0.9, val_frac=0.05, test_frac=0.05, random_state=42):
    assert train_frac + val_frac + test_frac == 1, "Fractions must sum to 1"
    
    # Split dataframe into train and temp (val + test) sets
    train_df, temp_df = train_test_split(df, train_size=train_frac, random_state=random_state)
    
    # Split temp set into validation and test sets
    val_size = val_frac / (val_frac + test_frac)
    val_df, test_df = train_test_split(temp_df, train_size=val_size, random_state=random_state)
    
    return train_df, val_df, test_df


def refinement_post_process_for_easing(df):
    secure_code = []
    vulnerable_code = []
    instruction = []

    for i in range(len(df)):
        temp_row = df.iloc[i]

        vul = temp_row["Vulnerable Code"].strip()
        sec = temp_row["Secure Code"].strip()
        more_sec = temp_row["More Secure Code"].strip()
        inst = temp_row["Instruction"].strip()

        if(more_sec!="Nothing"):
            secure_code.append(more_sec)
            vulnerable_code.append(vul)
            instruction.append(inst)
        else:
            secure_code.append(sec)
            vulnerable_code.append(vul)
            instruction.append(inst)
        
        if(secure_code[-1]==vulnerable_code[-1]):
            print("Dropped!")
            secure_code.pop()
            vulnerable_code.pop()
            instruction.pop()
        
    new_df =  pd.DataFrame({"Secure Code":secure_code,"Vulnerable Code":vulnerable_code,"Instruction":instruction})
    return new_df

In [2]:
#Load the dataframes
df_simple = pd.read_csv("datasets/simple_synth_data.csv")
df_complex = pd.read_csv("datasets/complex_synth_data.csv")

df_simple.head()

,Unnamed: 0,Prompt,Output
0,0,The following is a security issue found in Pyt...,VUL: \n```python\nfrom django.utils.safestring...
1,1,The following is a security issue found in Pyt...,VUL: \n```python\nimport os\n\ndef list_files(...
2,2,The following is a security issue found in Pyt...,VUL: \n```python\nimport subprocess\n\ndef exe...
3,3,The following is a security issue found in Pyt...,VUL: \n```python\nimport ftplib\n\ndef upload_...
4,4,The following is a security issue found in Pyt...,VUL: \n```python\nimport subprocess\n\ndef lis...


In [5]:
combined_gpt_output = df_simple["Output"].tolist() + df_complex["Output"].tolist()
parsed_df = parsed_dataframe(combined_gpt_output)
parsed_df.to_csv("datasets/initial_synth_data.csv")
parsed_df.head()



348 VUL: 
```python
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding, rsa

def verify_signature(public_key_pem, signature, message):
    public_key = serialization.load_pem_public_key(public_key_pem)
    try:
        public_key.verify(
            signature,
            message,
            padding.PKCS1v15(),
            hashes.SHA256()
        )
        return True
    except Exception:
        return False

public_key_pem = b"""-----BEGIN PUBLIC KEY-----
MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEA7v8Z5Q5Q5Q5Q5Q5Q5Q5Q
5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5
Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5
Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5
Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5
Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5
Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5Q5
Q5Q5

,Vulnerable Code,Secure Code,Vulnerable Code Reasoning,Secure Code Reasoning,Instruction
0,from django.utils.safestring import mark_safe\...,from django.shortcuts import render\nfrom djan...,The code directly marks user input as safe wit...,"The secure version escapes user input, prevent...",Create a Django view that processes user input...
1,import os\n\ndef list_files(directory):\n c...,import subprocess\n\ndef list_files(directory)...,The code is vulnerable because it directly con...,The secure version uses `subprocess` with a li...,Write a Python script that takes user input to...
2,import subprocess\n\ndef execute_command(user_...,import subprocess\n\ndef execute_command(user_...,The code is vulnerable because it uses `subpro...,The secure version avoids shell injection by p...,Write a Python function that takes user input ...
3,"import ftplib\n\ndef upload_file_ftp(server, u...",import paramiko\n\ndef upload_file_sftp(server...,This code is vulnerable because it uses the FT...,The second version is fixed because it uses th...,Write a Python function to upload a file to a ...
4,import subprocess\n\ndef list_files(directory)...,import subprocess\n\ndef list_files(directory)...,This code is vulnerable because it uses `shell...,The secure version avoids shell injection by p...,Write a Python script that lists files in a di...


In [5]:
#Create the final dataset and save them as train,test and validation

#Function to further post-process the dataset for ease of use

def refinement_post_process_for_easing(df):
    secure_code = []
    vulnerable_code = []
    instruction = []

    for i in range(len(df)):
        temp_row = df.iloc[i]

        vul = temp_row["Vulnerable Code"].strip()
        sec = temp_row["Secure Code"].strip()
        more_sec = temp_row["More Secure Code"].strip()
        inst = temp_row["Instruction"].strip()

        if(more_sec!="Nothing"):
            secure_code.append(more_sec)
            vulnerable_code.append(vul)
            instruction.append(inst)
        else:
            secure_code.append(sec)
            vulnerable_code.append(vul)
            instruction.append(inst)
        
        if(secure_code[-1]==vulnerable_code[-1]):
            print("Dropped!")
            secure_code.pop()
            vulnerable_code.pop()
            instruction.pop()
            

        # if(vul!=sec):
        #     secure_code.append(sec)
        #     vulnerable_code.append(vul)
        #     instruction.append(inst)
        
        # if(vul!=more_sec and more_sec!="Nothing"):
        #     secure_code.append(more_sec)
        #     vulnerable_code.append(vul)
        #     instruction.append(inst)
        
        # if(sec!=more_sec and more_sec!="Nothing"):
        #     secure_code.append(more_sec)
        #     vulnerable_code.append(sec)
        #     instruction.append(inst)
        
    new_df =  pd.DataFrame({"Secure Code":secure_code,"Vulnerable Code":vulnerable_code,"Instruction":instruction})
    return new_df


df_refined = pd.read_csv("datasets/final_synth_data.csv")


train_df,val_df,test_df = train_val_test_split(df_refined,train_frac=0.95, val_frac=0.03, test_frac=0.02)
train_df = train_df.reset_index()
val_df = val_df.reset_index()
test_df = test_df.reset_index()

train_df = refinement_post_process_for_easing(train_df)
val_df = refinement_post_process_for_easing(val_df)
test_df = refinement_post_process_for_easing(test_df)

train_df.to_csv("datasets/synth_train_refined.csv")
val_df.to_csv("datasets/synth_val_refined.csv")
test_df.to_csv("datasets/synth_test_refined.csv")

Dropped!
Dropped!
Dropped!
Dropped!
Dropped!
Dropped!
Dropped!
Dropped!
Dropped!
Dropped!
Dropped!
Dropped!
Dropped!


In [3]:
#Create the final dataset and save them as train,test and validation

#Function to further post-process the dataset for ease of use

def refinement_post_process_for_easing(df):
    secure_code = []
    vulnerable_code = []
    instruction = []

    for i in range(len(df)):
        temp_row = df.iloc[i]

        vul = temp_row["Vulnerable Code"].strip()
        sec = temp_row["Secure Code"].strip()
        more_sec = temp_row["More Secure Code"].strip()
        inst = temp_row["Instruction"].strip()

        if(more_sec!="Nothing"):
            secure_code.append(more_sec)
            vulnerable_code.append(vul)
            instruction.append(inst)
        else:
            secure_code.append(sec)
            vulnerable_code.append(vul)
            instruction.append(inst)
        
        if(secure_code[-1]==vulnerable_code[-1]):
            print("Dropped!")
            secure_code.pop()
            vulnerable_code.pop()
            instruction.pop()
            

        # if(vul!=sec):
        #     secure_code.append(sec)
        #     vulnerable_code.append(vul)
        #     instruction.append(inst)
        
        # if(vul!=more_sec and more_sec!="Nothing"):
        #     secure_code.append(more_sec)
        #     vulnerable_code.append(vul)
        #     instruction.append(inst)
        
        # if(sec!=more_sec and more_sec!="Nothing"):
        #     secure_code.append(more_sec)
        #     vulnerable_code.append(sec)
        #     instruction.append(inst)
        
    new_df =  pd.DataFrame({"Secure Code":secure_code,"Vulnerable Code":vulnerable_code,"Instruction":instruction})
    return new_df


df_refined = pd.read_csv("datasets/final_synth_data.csv")

#Dropping off everything from the complex dataset
df_refined = df_refined.iloc[:535].copy()

train_df = df_refined.iloc[:535].copy()
val_df = df_refined.iloc[535:635].copy()
test_df = df_refined.iloc[635:735].copy()

# train_df,val_df,test_df = train_val_test_split(df_refined,train_frac=0.95, val_frac=0.03, test_frac=0.02)
# train_df = train_df.reset_index()
# val_df = val_df.reset_index()
# test_df = test_df.reset_index()

train_df = refinement_post_process_for_easing(train_df)
val_df = refinement_post_process_for_easing(val_df)
test_df = refinement_post_process_for_easing(test_df)

train_df.to_csv("datasets/synth_train_refined_minus.csv")
val_df.to_csv("datasets/synth_val_refined_minus.csv")
test_df.to_csv("datasets/synth_test_refined_minus.csv")

Dropped!
Dropped!


In [ ]:
#Create the final dataset and save them as train,test and validation

#Function to further post-process the dataset for ease of use

def refinement_post_process_for_easing(df):
    secure_code = []
    vulnerable_code = []
    instruction = []

    for i in range(len(df)):
        temp_row = df.iloc[i]

        vul = temp_row["Vulnerable Code"].strip()
        sec = temp_row["Secure Code"].strip()
        more_sec = temp_row["More Secure Code"].strip()
        inst = temp_row["Instruction"].strip()

        if(more_sec!="Nothing"):
            secure_code.append(more_sec)
            vulnerable_code.append(vul)
            instruction.append(inst)
        else:
            secure_code.append(sec)
            vulnerable_code.append(vul)
            instruction.append(inst)
        
        if(secure_code[-1]==vulnerable_code[-1]):
            print("Dropped!")
            secure_code.pop()
            vulnerable_code.pop()
            instruction.pop()
        
    new_df =  pd.DataFrame({"Secure Code":secure_code,"Vulnerable Code":vulnerable_code,"Instruction":instruction})
    return new_df


df_refined = pd.read_csv("datasets/final_synth_data.csv")


train_df,val_df,test_df = train_val_test_split(df_refined,train_frac=0.95, val_frac=0.03, test_frac=0.02)
train_df = train_df.reset_index()
val_df = val_df.reset_index()
test_df = test_df.reset_index()

train_df = refinement_post_process_for_easing(train_df)
val_df = refinement_post_process_for_easing(val_df)
test_df = refinement_post_process_for_easing(test_df)

train_df.to_csv("datasets/synth_train_refined_minus.csv")
val_df.to_csv("datasets/synth_val_refined_minus.csv")
test_df.to_csv("datasets/synth_test_refined_minus.csv")